# Procurement & Supplier Management
**Student notebook**

## Procurement case
A manufacturer buys **VALVE20, a 20 mm brass valve**, from Suppliers A, B and C. All prices are in EUR per piece. The buyer is reviewing delivery, quality and recorded costs before planning a **9,000-unit July requirement**.

The data are synthetic. There are 60 closed orders per supplier, plus three status records per supplier. The review date is **30 June 2026**. The setup cell states the hypothetical July capacity and qualification status for each supplier.

Run setup and Baseline first, then Tasks 1–5. Run each task at its BEFORE value, inspect the output, change only the marked setting and run again. Each experiment reads the same source records and starts independently. Answer Notice, Think and Explain using the displayed evidence.


## Measures and their meaning
- **PO:** purchase order. One row is one order for one component. A closed row records its single final receipt; a short receipt closes with the unreceived balance cancelled.
- **Lead time:** calendar days from order to actual receipt. The mean describes average speed; standard deviation describes spread in days.
- **Promised-date OTD:** on-time delivery percentage among closed orders with known receipts. Receipt on the promised date counts as on time. Open and unconfirmed records are shown separately.
- **In full:** received quantity is at least ordered quantity. Timeliness and completeness are separate checks.
- **Flagged-order percentage:** orders containing at least one defective unit / inspected closed orders.
- **Defective-unit percentage:** defective units / inspected units. Every received unit in a closed order is inspected in this case.
- **Recorded procurement cost:** purchase, freight, inspection and rework divided by received units. Rework restores defective units; defects remain recorded as first-inspection failures. No purchase refund is assumed.
- **Qualification:** approval to supply this component. A performance screen then applies the buyer’s stated requirements. Capacity is the assumed available quantity for July.


## Setup
Run the following folded cells. An internet connection is required: the purchase-order data are loaded from the GitHub CSV. Supplier approval and July capacity are stated case assumptions. Task settings appear below Baseline.

In [ ]:
"""Calculations for the synthetic Hof procurement case."""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
REVIEW_DATE=pd.Timestamp('2026-06-30')


In [ ]:
import pandas as pd

DATA_URL = 'https://raw.githubusercontent.com/saikisri97/17_Hof_Lecture_Code_Pingo/refs/heads/main/Supply_Chain_Analytics/data/procurement_po_data.csv'
orders_raw = pd.read_csv(DATA_URL)

# Check that GitHub contains the revised procurement dataset.
required_columns = ['PO_ID', 'Supplier', 'Product_Code', 'Product', 'Unit', 'Currency', 'PO_Date', 'Promised_Date', 'Actual_Receipt_Date', 'Status', 'Ordered_Qty', 'Received_Qty', 'Inspected_Qty', 'Defective_Qty', 'Unit_Price_EUR', 'Freight_EUR', 'Inspection_Cost_EUR', 'Rework_Cost_EUR']
missing_columns = sorted(set(required_columns) - set(orders_raw.columns))
if missing_columns:
    raise ValueError(
        "The GitHub CSV does not match this notebook. Upload the revised "
        "procurement_po_data.csv to Supply_Chain_Analytics/data/ and rerun Setup. "
        f"Missing columns: {missing_columns}"
    )

# Supplier approval and July capacity are stated case assumptions.
suppliers = pd.DataFrame([
    ['A', 'Approved', 8000],
    ['B', 'Approved', 6000],
    ['C', 'Conditional', 12000],
], columns=['Supplier', 'Qualification', 'Available_Capacity_July_units'])
print(f"Loaded {len(orders_raw)} purchase orders from GitHub.")


In [ ]:
def prepare(orders):
    d=orders.copy()
    for col in ['PO_Date','Promised_Date','Actual_Receipt_Date']:
        d[col]=pd.to_datetime(d[col],errors='raise')
    d['Lead_Days']=(d.Actual_Receipt_Date-d.PO_Date).dt.days
    d['Days_After_Promise']=(d.Actual_Receipt_Date-d.Promised_Date).dt.days
    return d

In [ ]:
def completed(orders):
    return orders.loc[orders.Status.eq('Closed') & orders.Actual_Receipt_Date.notna()].copy()

In [ ]:
def supplier_metrics(orders):
    rows=[]
    for supplier,g in completed(orders).groupby('Supplier'):
        received=g.Received_Qty.sum();inspected=g.Inspected_Qty.sum()
        purchase=(g.Received_Qty*g.Unit_Price_EUR).sum()
        extra=g[['Freight_EUR','Inspection_Cost_EUR','Rework_Cost_EUR']].sum().sum()
        rows.append({'Supplier':supplier,'Completed POs':len(g),'Mean days':g.Lead_Days.mean(),
          'Std days':g.Lead_Days.std(ddof=1),'Promised-date OTD %':g.Days_After_Promise.le(0).mean()*100,
          'In-full %':g.Received_Qty.ge(g.Ordered_Qty).mean()*100,
          'Flagged POs':int(g.Defective_Qty.gt(0).sum()),'Flagged PO %':g.Defective_Qty.gt(0).mean()*100,
          'Inspected units':int(inspected),'Defective units':int(g.Defective_Qty.sum()),
          'Defective unit %':g.Defective_Qty.sum()/inspected*100,
          'Purchase EUR/unit':purchase/received,'Recorded cost EUR/unit':(purchase+extra)/received})
    return pd.DataFrame(rows).set_index('Supplier')

In [ ]:
def baseline(orders,profiles):
    display(orders.head(3));display(profiles)
    print('Review date: 30 June 2026. Synthetic case: VALVE20, 20 mm brass valve, pieces, EUR.')
    table=orders.groupby(['Supplier','Status']).size().unstack(fill_value=0);display(table)
    open_orders=orders[orders.Status.eq('Open')]
    print(f'Open overdue: {int(open_orders.Promised_Date.lt(REVIEW_DATE).sum())}; open not yet due: {int(open_orders.Promised_Date.ge(REVIEW_DATE).sum())}; receipt unconfirmed: {int(orders.Status.eq("Receipt_unconfirmed").sum())}.')
    print('The completed-order scorecard uses 60 orders per supplier. Open and unconfirmed orders remain in this separate status view.')

In [ ]:
def show_delivery(orders,supplier='A'):
    if supplier not in ['A','B']:raise ValueError('Choose A or B.')
    g=completed(orders);m=supplier_metrics(orders)
    display(m.loc[['A','B'],['Completed POs','Mean days','Std days']].round(2))
    fig,ax=plt.subplots(figsize=(8,3));vals=g.loc[g.Supplier.eq(supplier),'Lead_Days']
    ax.hist(vals,bins=np.arange(4.5,14.6,1),color='#5B9BD5',edgecolor='white');ax.set(xlabel='Calendar days from order to receipt',ylabel='Completed orders',title=f'Supplier {supplier}: 60 completed orders',xlim=(4.5,14.5),ylim=(0,26));plt.tight_layout();plt.show()
    print('Both runs use the same date range and fixed plot scales. Standard deviation summarises spread around the mean in days.')
    return m.loc[supplier]

In [ ]:
def show_target(orders,target_days=10):
    if target_days not in [10,12]:raise ValueError('Choose 10 or 12 days.')
    g=completed(orders);m=supplier_metrics(orders)
    count=g.groupby('Supplier').Lead_Days.apply(lambda x:int(x.le(target_days).sum()))
    result=pd.DataFrame({'Completed POs':m['Completed POs'],'Within internal target':count,'Within target %':count/m['Completed POs']*100,'Promised-date OTD %':m['Promised-date OTD %']})
    display(result.round(2));result[['Within target %','Promised-date OTD %']].plot.bar(figsize=(8,3),ylim=(0,100),color=['#5B9BD5','#ED7D31']);plt.ylabel('Completed orders (%)');plt.title(f'Internal target: {target_days} calendar days');plt.xticks(rotation=0);plt.tight_layout();plt.show()
    print('The internal target is a hypothetical reporting rule. Original promised dates and actual receipts stay unchanged.')
    return result

In [ ]:
def show_quality(orders,view='orders'):
    if view not in ['orders','units']:raise ValueError('Choose orders or units.')
    m=supplier_metrics(orders);display(m[['Completed POs','Flagged POs','Flagged PO %','Inspected units','Defective units','Defective unit %']].round(2))
    col='Flagged PO %' if view=='orders' else 'Defective unit %'
    m[col].plot.bar(figsize=(7,3),color='#5B9BD5',ylim=(0,35 if view=='orders' else 1.4));plt.ylabel(col);plt.title('Orders with any defect' if view=='orders' else 'Defective units among inspected units');plt.xticks(rotation=0);plt.tight_layout();plt.show()
    print('All received units in the closed orders were inspected. A flagged order may contain one or many defective units. Read the axis: the two charts use different scales.')
    return m

In [ ]:
def show_cost(orders,basis='purchase'):
    if basis not in ['purchase','recorded']:raise ValueError('Choose purchase or recorded.')
    m=supplier_metrics(orders);display(m[['Purchase EUR/unit','Recorded cost EUR/unit']].round(3))
    col='Purchase EUR/unit' if basis=='purchase' else 'Recorded cost EUR/unit'
    m[col].plot.bar(figsize=(7,3),color='#5B9BD5',ylim=(0,12));plt.ylabel('EUR per received unit');plt.title('Purchase price' if basis=='purchase' else 'Recorded procurement cost');plt.xticks(rotation=0);plt.tight_layout();plt.show()
    print('Recorded cost = purchase + freight + inspection + rework, divided by received units. These case records exclude downtime, financing, administration and later lifecycle costs.')
    return m

In [ ]:
def screen_suppliers(orders,profiles,max_defective_pct=1.0):
    if max_defective_pct not in [1.0,.5]:raise ValueError('Choose 1.0 or 0.5 percent.')
    m=supplier_metrics(orders).join(profiles.set_index('Supplier'))
    m['Approved']=m.Qualification.eq('Approved');m['Delivery passes']=m['Promised-date OTD %'].ge(85);m['Quality passes']=m['Defective unit %'].le(max_defective_pct)
    m['Eligible']=m[['Approved','Delivery passes','Quality passes']].all(axis=1)
    display(m[['Qualification','Promised-date OTD %','Defective unit %','Available_Capacity_July_units','Delivery passes','Quality passes','Eligible']].round(2))
    capacity=int(m.loc[m.Eligible,'Available_Capacity_July_units'].sum());need=9000
    print(f'July requirement: {need:,} units. Eligible capacity: {capacity:,}. Capacity gap: {max(0,need-capacity):,}.')
    print('Hypothetical buyer rules: Approved, promised-date OTD at least 85%, and the chosen defective-unit ceiling. Capacity is assumed available for July, not an allocation recommendation.')
    print('A follow-up should identify the failed criterion, request evidence and agree a measurable review. Changing a rule does not improve a supplier.')
    return m

## Baseline
Read the first records and the order-status counts. A blank receipt date is interpreted together with the order status.

In [ ]:
orders = prepare(orders_raw)
baseline(orders, suppliers)

**First record:** A-001 orders 400 VALVE20 units on 2 January. Supplier A promises 13 January and delivers 380 units on 11 January. All 380 are inspected; 6 initially fail inspection and are reworked. The order arrives on time but 20 units short. Purchase price is EUR 9.00 per received unit.

## 1. Compare delivery speed and spread

**BEFORE:** Run Supplier A. Read its mean, spread and delivery-time histogram.

**CHANGE:** Change SUPPLIER from A to B. Keep the same completed orders and date range.

**AFTER:** Compare the histograms using their fixed axes and the two supplier rows.

In [ ]:
SUPPLIER = 'B'  # <-- CHANGE HERE: 'A' -> 'B'
result_1 = show_delivery(orders, supplier=SUPPLIER)  # <-- READ OUTPUT

**THINK**

1. **Notice:** Which supplier is faster on average, and which has a wider spread of delivery times?

2. **Think:** A buyer says, “B arrives faster, so its timing is more predictable.” Do both parts follow from these outputs?

3. **Explain:** Explain the difference between speed and consistency to the buyer using one value for each.

## 2. Read delivery performance against a target

**BEFORE:** Use the internal target of 10 days. Read the target percentage and promised-date OTD.

**CHANGE:** Change TARGET_DAYS from 10 to 12. Original promises and actual receipts stay fixed.

**AFTER:** Compare the internal-target bars with the promised-date bars.

In [ ]:
TARGET_DAYS = 10  # <-- CHANGE HERE: 10 -> 12
result_2 = show_target(orders, target_days=TARGET_DAYS)  # <-- READ OUTPUT

**THINK**

1. **Notice:** Which percentages change when the internal target moves from 10 to 12 days?

2. **Think:** The reported target percentage improves. Have any suppliers actually delivered sooner?

3. **Explain:** A colleague wants to tell management that delivery performance improved. Explain what this run really changed.

## 3. Compare two quality measures

**BEFORE:** Read the percentage of completed orders containing any defective unit.

**CHANGE:** Change QUALITY_VIEW from orders to units.

**AFTER:** Read both denominators in the table and compare the A/B ranking. The chart scale changes with the measure.

In [ ]:
QUALITY_VIEW = 'orders'  # <-- CHANGE HERE: 'orders' -> 'units'
result_3 = show_quality(orders, view=QUALITY_VIEW)  # <-- READ OUTPUT

**THINK**

1. **Notice:** Compare A and B: which has fewer flagged orders, and which has the lower defective-unit percentage?

2. **Think:** Why can B have fewer flagged orders but a larger defective-unit percentage? Use the counts in the table.

3. **Explain:** A manager calls B “better quality” using only flagged orders. Explain what the second measure adds.

## 4. Compare purchase price and recorded cost

**BEFORE:** Compare purchase prices for the same VALVE20 component in EUR per piece.

**CHANGE:** Change COST_BASIS from purchase to recorded.

**AFTER:** Compare purchase-only and recorded procurement cost per received unit.

In [ ]:
COST_BASIS = 'purchase'  # <-- CHANGE HERE: 'purchase' -> 'recorded'
result_4 = show_cost(orders, basis=COST_BASIS)  # <-- READ OUTPUT

**THINK**

1. **Notice:** Which supplier has the lowest purchase price? Which has the lowest recorded cost per received unit?

2. **Think:** What included costs explain why the ranking changes?

3. **Explain:** A colleague says the cheapest quoted supplier must cost the least overall. Explain what the cost table supports and one cost it still leaves out.

## 5. Screen suppliers and check capacity

**BEFORE:** Apply the 1.0% defective-unit ceiling, Approved status and at least 85% promised-date OTD. July need is 9,000 units.

**CHANGE:** Change MAX_DEFECTIVE_PCT from 1.0 to 0.5. Keep all other rules and capacity figures fixed.

**AFTER:** Read each pass/fail column, the eligible suppliers and total available July capacity.

In [ ]:
MAX_DEFECTIVE_PCT = 1.0  # <-- CHANGE HERE: 1.0 -> 0.5
result_5 = screen_suppliers(orders, suppliers, max_defective_pct=MAX_DEFECTIVE_PCT)  # <-- READ OUTPUT

**THINK**

1. **Notice:** Which suppliers qualify in each run, and what happens to capacity relative to the July requirement?

2. **Think:** Did the stricter quality ceiling change supplier quality or the buyer’s acceptance rule?

3. **Explain:** Write a two-sentence follow-up for procurement: explain why B drops out, what to review with B, and why the remaining shortlist cannot cover July on its own.

## Supplier review record
Record the product, period, measure and denominator beside each finding. A supplier follow-up states the issue, the evidence requested, the agreed action and the next review measure.

The purchasing plan passes expected receipt dates, quantities and reliability information to inventory planning.